# 03 — Preparación y esquema de validación

Este paso separa el holdout interno antes de modelar y define el preprocesamiento que se ajustará dentro de cada fold. El test de competencia permanece aparte y no se usa para medir desempeño.

## Diseño

- Solo las filas `source_split == "train"` con `Exited` binario participan en el desarrollo.
- Se reserva un holdout estratificado del 20% con semilla fija. No se usa para escoger features, ajustar hiperparámetros ni comparar modelos; se evalúa una sola vez al final.
- El 80% restante se usa en validación cruzada estratificada de 3 folds. En cada fold se ajustan imputadores, escalador y categorías one-hot únicamente con la porción de entrenamiento.
- El `Pipeline` coloca la ingeniería de features y el `ColumnTransformer` antes del estimador, para que la validación cruzada ajuste todo el preprocesamiento sin fuga.
- Las filas del test de competencia no son un holdout: no tienen etiquetas y se conservan separadas para predicciones futuras.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "datos.py").exists():
    raise FileNotFoundError("Abre este notebook desde la carpeta PSet3.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.datos import load_obt
from src.pipeline import make_model_pipeline
from src.preparacion import make_stratified_cv, split_churn_data

pd.set_option("display.max_columns", 50)

In [ ]:
obt = load_obt()
obt.columns = obt.columns.str.lower()
split = split_churn_data(obt)

assert "exited" not in split.X_development.columns
assert "exited" not in split.X_holdout.columns
assert "exited" not in split.X_competition_test.columns
assert set(split.y_development.unique()) == {0, 1}
assert set(split.y_holdout.unique()) == {0, 1}

split_summary = pd.DataFrame(
    [
        {
            "particion": "desarrollo",
            "filas": len(split.X_development),
            "churn": int(split.y_development.sum()),
            "tasa_churn_pct": 100 * split.y_development.mean(),
        },
        {
            "particion": "holdout",
            "filas": len(split.X_holdout),
            "churn": int(split.y_holdout.sum()),
            "tasa_churn_pct": 100 * split.y_holdout.mean(),
        },
        {
            "particion": "test_competencia_sin_etiqueta",
            "filas": len(split.X_competition_test),
            "churn": None,
            "tasa_churn_pct": None,
        },
    ]
)
display(split_summary.round(2))
print("Features disponibles:", split.X_development.columns.tolist())

## Validación cruzada sin fuga

Los folds se generan exclusivamente dentro de desarrollo. Las proporciones de churn permiten revisar la estratificación; el holdout no participa en ellos.

In [ ]:
cv = make_stratified_cv()
fold_rows = []
for fold_number, (train_indices, validation_indices) in enumerate(
    cv.split(split.X_development, split.y_development), start=1
):
    y_fold_train = split.y_development.iloc[train_indices]
    y_fold_validation = split.y_development.iloc[validation_indices]
    assert set(train_indices).isdisjoint(validation_indices)
    fold_rows.append(
        {
            "fold": fold_number,
            "filas_entrenamiento": len(train_indices),
            "filas_validacion": len(validation_indices),
            "churn_entrenamiento_pct": 100 * y_fold_train.mean(),
            "churn_validacion_pct": 100 * y_fold_validation.mean(),
        }
    )

display(pd.DataFrame(fold_rows).round(2))

## Pipeline de preprocesamiento

Los valores numéricos de Snowflake llegan como `Decimal`; `ChurnFeatureEngineer` los convierte a tipo numérico antes de que `ColumnTransformer` seleccione columnas. Los numéricos usan imputación por mediana y escalado estándar; las categorías, imputación por moda y `OneHotEncoder(handle_unknown="ignore")`. El estimador de abajo es solo un marcador estructural: los modelos y sus métricas se comparan en el paso siguiente.

In [ ]:
pipeline_template = make_model_pipeline(DummyClassifier(strategy="prior"))
display(pipeline_template)
print("La validación cruzada ajustará cada etapa con cada fold de entrenamiento.")